In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_excel(r"C:\Users\BM\OneDrive\Desktop\Python_projects\DS\customer_segmentation_intervention\data\raw\online+retail+ii\online_retail_II.xlsx", sheet_name="Year 2010-2011")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df.shape

(541910, 8)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [7]:
df.isna().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [14]:
# Cancelled invoices

df[df['Invoice'].astype(str).str.startswith("C")].shape

(9288, 8)

In [15]:
# Negative quantities

df[df['Quantity'] < 0].shape

(10624, 8)

In [16]:
df[df['Price'] < 0].shape

(2, 8)

In [17]:
print(df['InvoiceDate'].min())
print(df['InvoiceDate'].max())

2010-12-01 08:26:00
2011-12-09 12:50:00


In [18]:
df["Quantity"].describe()

count    541910.000000
mean          9.552234
std         218.080957
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [19]:
df["Price"].describe()

count    541910.000000
mean          4.611138
std          96.759765
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: Price, dtype: float64

In [20]:
df["Country"].value_counts().head(10)

Country
United Kingdom    495478
Germany             9495
France              8558
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

In [22]:
print(df[df["Quantity"] == -80995])

        Invoice StockCode                  Description  Quantity  \
540422  C581484     23843  PAPER CRAFT , LITTLE BIRDIE    -80995   

               InvoiceDate  Price  Customer ID         Country  
540422 2011-12-09 09:27:00   2.08      16446.0  United Kingdom  


In [24]:
df[df['Invoice'].astype(str).str.startswith('A')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


## Data Quality Assessment

Raw dataset: 541,910 rows, 8 columns, covering Dec 2010 to Dec 2011.

Cleaning decisions:
- Drop rows with null Customer ID (135,080 rows): RFM requires longitudinal 
  customer tracking. Anonymous sessions are excluded by design.
- Drop cancelled invoices (C prefix): Net of cancellations is handled by 
  excluding these rows entirely.
- Drop rows where Quantity <= 0 or Price <= 0: These represent anomalies 
  and bad debt adjustments, not real transactions.

Snapshot date for Recency: 2011-12-10 (one day after last transaction).

In [25]:
df_clean = df.copy()

In [27]:
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')] # Cancellation removed
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('A')] # Bad debts removed
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['Price'] > 0]

In [29]:
df_clean = df_clean[df_clean['Customer ID'].notna()]

In [30]:
df_clean.shape

(397885, 8)

In [32]:
# Fix customer ID data type

df_clean['Customer ID'] = df_clean['Customer ID'].astype(int).astype(str)

In [33]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397885 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      397885 non-null  object        
 1   StockCode    397885 non-null  object        
 2   Description  397885 non-null  object        
 3   Quantity     397885 non-null  int64         
 4   InvoiceDate  397885 non-null  datetime64[ns]
 5   Price        397885 non-null  float64       
 6   Customer ID  397885 non-null  object        
 7   Country      397885 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 27.3+ MB


In [34]:
# Create revenue column

df_clean['Revenue'] = df_clean['Price'] * df_clean['Quantity']

In [35]:
df_clean.Revenue.describe()

count    397885.000000
mean         22.396989
std         309.070653
min           0.001000
25%           4.680000
50%          11.800000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64

In [40]:
df_clean.to_csv('../data/processed/clean_transactions.csv', index=False)

## Output
Clean dataset: 397,885 rows saved to data/processed/clean_transactions.csv
Snapshot date: 2011-12-10
Ready for RFM feature engineering in notebook 02.